# Emotion Recognition System — Dataset Quality Analysis

**Internship:** NIT Sikkim  
This notebook audits the provided YOLO dataset before preprocessing or model training. It treats `train`, `valid`, and `test` as separate splits.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import math

import cv2
import numpy as np
import pandas as pd
import yaml
from PIL import Image

pd.set_option('display.max_colwidth', None)

In [ ]:
dataset_path = Path('../dataset/YOLO_format').resolve()
if not dataset_path.exists():
    dataset_path = Path('dataset/YOLO_format').resolve()

with (dataset_path / 'data.yaml').open(encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

class_names = data_config['names']
if isinstance(class_names, dict):
    class_names = [class_names[index] for index in sorted(class_names)]

splits = {
    name: {'images': dataset_path / name / 'images', 'labels': dataset_path / name / 'labels'}
    for name in ('train', 'valid', 'test')
}

print('Dataset:', dataset_path)
print('Classes:', dict(enumerate(class_names)))
for name, folders in splits.items():
    print(f"{name}: images={folders['images'].exists()}, labels={folders['labels'].exists()}")

## 1. Dataset structure

The configured folders and emotion-class mapping are displayed above. The following commits add image, label, duplication, and sharpness quality checks.

## 2. Image properties and resolution

Every image is decoded to check readability while recording dimensions, format, colour mode, channels, and file size.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def image_files(folder):
    return sorted(path for path in folder.iterdir()
                  if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)

def analyze_images(paths):
    result = {'total_images': len(paths), 'readable_images': 0, 'corrupt_images': 0,
              'sizes': Counter(), 'formats': Counter(), 'modes': Counter(),
              'channels': Counter(), 'file_sizes_bytes': [], 'records': [], 'errors': []}
    for path in paths:
        try:
            with Image.open(path) as image:
                image.load()
                width, height = image.size
                mode = image.mode
                channels = len(image.getbands())
                result['readable_images'] += 1
                result['sizes'][(width, height)] += 1
                result['formats'][image.format or 'Unknown'] += 1
                result['modes'][mode] += 1
                result['channels'][channels] += 1
                result['file_sizes_bytes'].append(path.stat().st_size)
                result['records'].append({'path': path, 'width': width, 'height': height,
                                          'mode': mode, 'channels': channels,
                                          'format': image.format or 'Unknown',
                                          'file_size_bytes': path.stat().st_size})
        except Exception as error:
            result['corrupt_images'] += 1
            result['errors'].append((path.name, str(error)))
    return result

split_images = {name: image_files(folders['images']) for name, folders in splits.items()}
image_analysis = {name: analyze_images(paths) for name, paths in split_images.items()}

for name, result in image_analysis.items():
    print(f'\n{name.upper()}')
    print('Total/readable/corrupt:', result['total_images'], result['readable_images'], result['corrupt_images'])
    print('Resolutions:', dict(result['sizes']))
    print('Formats:', dict(result['formats']))
    print('Modes:', dict(result['modes']))
    print('Channels:', dict(result['channels']))